# Namma Space — Autonomous 3D Gaussian Splatting Reconstruction Pipeline
**Techfest IIT Bombay 2026–27 | Namma Space Challenge (Round 1)**

This notebook trains your own authentic, end-to-end 3D Gaussian Splatting digital twin from your smartphone walkthrough video using an NVIDIA T4 GPU on Google Colab.

---
### ⚡ 2-Step Quick Start:
1. **GPU Check**: In the top menu, go to **Runtime → Change runtime type** → select **T4 GPU** → click **Save**.
2. **Run Step 1**: Prepares Python 3.10 with official CUDA 11.8 runtime binaries and installs precompiled rasterizers (Instant — zero compilation!).
3. **Run Step 2**: Discovers your uploaded video (`.mp4`), extracts frames, computes 3D camera poses via COLMAP, trains 3DGS (30k iterations), and automatically downloads `namma_space_assets.zip` to your laptop!

### Step 1: Automated Environment Setup & Instant CUDA Rasterizer Load
Installs Python 3.10 bundled with official CUDA 11.8 runtime libraries (`cudatoolkit=11.8`), completely bypassing CUDA 13 compiler incompatibility, and loads precompiled rasterizers instantly.

In [ ]:
%%bash
set -e

echo "=== 1. Checking NVIDIA GPU ==="
nvidia-smi

echo "=== 2. Creating Python 3.10 Environment ==="
cd /content
if [ ! -f /content/bin/micromamba ]; then
    wget -qO- https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
fi

if [ ! -d /content/py310 ]; then
    ./bin/micromamba create -p /content/py310 python=3.10 -y -c conda-forge
fi

echo "=== 3. Installing PyTorch 2.0.1 (cu118 ABI Match) and Official CUDA 11 Runtime ==="
/content/py310/bin/pip install -q torch==2.0.1+cu118 torchvision==0.15.2+cu118 --index-url https://download.pytorch.org/whl/cu118
/content/py310/bin/pip install -q nvidia-cuda-runtime-cu11 "numpy<2"
/content/py310/bin/pip install -q setuptools wheel ninja plyfile tqdm pycolmap typer rich opencv-python-headless scipy

# Register CUDA 11 runtime (libcudart.so.11.0) globally in OS dynamic linker
CUDART_DIR=$(find /content/py310 -name "libcudart.so.11*" -exec dirname {} \; 2>/dev/null | head -n 1)
if [ -n "$CUDART_DIR" ]; then
    echo "Registering CUDA 11 runtime from: $CUDART_DIR"
    cp -P "$CUDART_DIR"/libcudart.so* /usr/lib/x86_64-linux-gnu/ 2>/dev/null || true
    cp -P "$CUDART_DIR"/libcudart.so* /content/py310/lib/ 2>/dev/null || true
    ldconfig 2>/dev/null || true
fi

echo "=== 4. Cloning Pipeline Repositories ==="
cd /content
if [ ! -d "/content/roomsplat" ]; then
    git clone -q https://github.com/Ahmedn1/roomsplat.git
fi
if [ ! -d "/content/gaussian-splatting" ]; then
    git clone -q --recursive https://github.com/camenduru/gaussian-splatting.git /content/gaussian-splatting
fi

# Patch dataset_readers.py to gracefully support all camera models
/content/py310/bin/python -c "
p = '/content/gaussian-splatting/scene/dataset_readers.py'
try:
    s = open(p).read()
    old = 'assert False, \"Colmap camera model not handled: only undistorted datasets (PINHOLE or SIMPLE_PINHOLE cameras) supported!\"'
    new = 'focal_length_x = intr.params[0]; focal_length_y = intr.params[1] if len(intr.params) > 1 else intr.params[0]; FovY = focal2fov(focal_length_y, height); FovX = focal2fov(focal_length_x, width)'
    if old in s:
        open(p, 'w').write(s.replace(old, new))
        print('[PASS] Patched dataset_readers.py for universal camera model support')
except Exception as e:
    print(f'Patch notice: {e}')
"

cd /content/roomsplat
/content/py310/bin/pip install -q -e .

echo "=== 5. Installing Precompiled CUDA Rasterizers (Instant — Zero Compilation) ==="
export LD_LIBRARY_PATH=/content/py310/lib:$LD_LIBRARY_PATH
/content/py310/bin/pip install --force-reinstall --no-deps https://huggingface.co/camenduru/gaussian-splatting/resolve/main/diff_gaussian_rasterization-0.0.0-cp310-cp310-linux_x86_64.whl
/content/py310/bin/pip install --force-reinstall --no-deps https://huggingface.co/camenduru/gaussian-splatting/resolve/main/simple_knn-0.0.0-cp310-cp310-linux_x86_64.whl

echo "=== 6. Verification Self-Test ==="
export LD_LIBRARY_PATH=/content/py310/lib:$LD_LIBRARY_PATH
/content/py310/bin/python -c "
import torch
assert torch.cuda.is_available(), 'CUDA not detected on PyTorch!'
import diff_gaussian_rasterization
import simple_knn
import pycolmap
import numpy as np
assert int(np.__version__.split('.')[0]) < 2, f'NumPy must be 1.x, got {np.__version__}'
print(f'\n[PASS] PyTorch {torch.__version__} with GPU: {torch.cuda.get_device_name(0)}')
print(f'[PASS] NumPy version: {np.__version__} (1.x binary compatibility confirmed)')
print('[PASS] diff_gaussian_rasterization: Ready (Zero Compilation)')
print('[PASS] simple_knn: Ready (Zero Compilation)')
print('[PASS] pycolmap: Ready')
print('\n' + '='*65)
print('  SUCCESS: PIPELINE IS 100% READY FOR TRAINING!')
print('  Run Step 2 below to process your video.')
print('='*65)
"


### Step 2: 1-Click Video Reconstruction & Web Twin Download
Discovers your uploaded video (or prompts you to select one), extracts frames, solves camera poses, trains 3DGS for 30,000 iterations, and downloads `namma_space_assets.zip` to your computer.

In [ ]:
import os, sys, glob, shutil
from pathlib import Path
from google.colab import files

# Expose CUDA 11 runtime libraries to all subprocesses
os.environ["LD_LIBRARY_PATH"] = "/content/py310/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

# 1. Enforce NumPy 1.x binary compatibility with PyTorch 2.0.1
os.system("/content/py310/bin/pip install -q 'numpy<2'")

# 2. Universal Camera Model Patch (applied directly in Python)
reader_path = Path("/content/gaussian-splatting/scene/dataset_readers.py")
if reader_path.exists():
    code = reader_path.read_text(encoding="utf-8")
    old_assert = 'assert False, "Colmap camera model not handled: only undistorted datasets (PINHOLE or SIMPLE_PINHOLE cameras) supported!"'
    patch_code = 'focal_length_x = intr.params[0]; focal_length_y = intr.params[1] if len(intr.params) > 1 else intr.params[0]; FovY = focal2fov(focal_length_y, height); FovX = focal2fov(focal_length_x, width)'
    if old_assert in code:
        reader_path.write_text(code.replace(old_assert, patch_code), encoding="utf-8")
        print("[PASS] Universal camera model patch applied to dataset_readers.py")

# 3. Auto-detect uploaded video in /content
video_candidates = [
    f for f in glob.glob("/content/*")
    if f.lower().endswith((".mp4", ".mov", ".m4v", ".avi")) and not f.startswith("/content/sample_data")
]

if not video_candidates:
    print("No video found in /content. Please select your room walkthrough video to upload:")
    uploaded = files.upload()
    video_candidates = list(uploaded.keys())
    if not video_candidates:
        raise FileNotFoundError("No video provided. Please upload an MP4/MOV room walkthrough.")

video_file = Path(video_candidates[0]).resolve()
video_size_mb = video_file.stat().st_size / (1024 * 1024)
print(f"\nTarget Video: {video_file.name} ({video_size_mb:.2f} MB)")

# Define output directories
out_dir = Path(f"/content/{video_file.stem}_splat")
frames_dir = out_dir / "frames"
images_dir = out_dir / "images"
sparse_dir = out_dir / "sparse" / "0"
model_dir = out_dir / "model"
viewer_dir = out_dir / "viewer"
viewer_dir.mkdir(parents=True, exist_ok=True)

# Clear previous partial sparse/model directories to re-solve with exhaustive matching
if not (model_dir / "point_cloud").exists():
    shutil.rmtree(sparse_dir, ignore_errors=True)
    shutil.rmtree(model_dir, ignore_errors=True)

# =============================================================================
# STAGE 1: Frame Extraction & Camera Pose Estimation (SIMPLE_PINHOLE + Exhaustive)
# =============================================================================
if not (sparse_dir / "cameras.bin").exists() and not (sparse_dir / "cameras.txt").exists():
    print("\n--- STAGE 1: Extracting Frames & Estimating Camera Trajectory (COLMAP) ---")
    stage1_py = f"""import sys, shutil
from pathlib import Path
sys.path.insert(0, '/content/roomsplat')
from roomsplat.pipeline.extract import extract_frames
from roomsplat.pipeline.poses import estimate_poses

out_dir = Path('{out_dir}')
frames_dir = Path('{frames_dir}')
images_dir = Path('{images_dir}')
video_file = Path('{video_file}')

if not frames_dir.exists() or len(list(frames_dir.glob('*.jpg'))) == 0:
    print('Extracting frames at 1.5 FPS (max dimension 1600)...')
    extract_frames(video_path=video_file, output_dir=frames_dir, target_fps=1.5, max_dim=1600)

if not images_dir.exists():
    images_dir.mkdir(parents=True, exist_ok=True)
    for f in sorted(frames_dir.glob('*.jpg')):
        shutil.copy2(f, images_dir / f.name)

print('Estimating camera poses via pycolmap (SIMPLE_PINHOLE, exhaustive matcher)...')
estimate_poses(
    frames_dir=images_dir,
    output_dir=out_dir,
    method='colmap',
    sequential_overlap=15,
    camera_model='SIMPLE_PINHOLE',
    undistort=False,
    matcher='exhaustive'
)
print('Pose estimation successfully finished!')
"""
    Path("/content/run_stage1.py").write_text(stage1_py)
    ret = os.system("/content/py310/bin/python /content/run_stage1.py")
    if ret != 0:
        raise RuntimeError(f"Stage 1 failed with return code {ret}")
else:
    print(f"\nReusing existing camera poses in: {sparse_dir}")

# =============================================================================
# STAGE 2: 3D Gaussian Splatting Training (30,000 Iterations on T4 GPU)
# =============================================================================
print("\n--- STAGE 2: Training 3D Gaussian Radiance Field (30k Iterations on T4 GPU) ---")
train_cmd = f"cd /content/gaussian-splatting && /content/py310/bin/python train.py -s '{out_dir}' -m '{model_dir}' --iterations 30000 --save_iterations 7000 30000 --densify_until_iter 8000 --densification_interval 200"
ret = os.system(train_cmd)
if ret != 0:
    raise RuntimeError(f"Training failed with return code {ret}")

# =============================================================================
# STAGE 3: Convert to 32-Byte scene.splat & Export cameras.json
# =============================================================================
print("\n--- STAGE 3: Packaging Web-Optimized Binary 3D Twin ---")
export_script = f"""import sys, shutil
from pathlib import Path
sys.path.insert(0, '/content/roomsplat')
from roomsplat.pipeline.export import ply_to_splat, write_cameras_json

out_dir = Path('{out_dir}')
model_dir = Path('{model_dir}')
sparse_dir = out_dir / 'sparse' / '0'
viewer_dir = out_dir / 'viewer'
viewer_dir.mkdir(parents=True, exist_ok=True)

ply_files = list(model_dir.glob('point_cloud/iteration_*/point_cloud.ply'))
if not ply_files:
    raise FileNotFoundError(f'No point_cloud.ply found in {model_dir}')
ply_path = sorted(ply_files, key=lambda p: int(p.parent.name.split('_')[1]))[-1]
print(f'Found trained point cloud: {ply_path}')

splat_path = viewer_dir / 'scene.splat'
ply_to_splat(ply_path, splat_path)
write_cameras_json(sparse_dir, viewer_dir)

shutil.copy(splat_path, '/content/scene.splat')
shutil.copy(viewer_dir / 'cameras.json', '/content/cameras.json')
print('Successfully exported scene.splat & cameras.json')
"""

Path("/content/export_bundle.py").write_text(export_script)
ret = os.system("/content/py310/bin/python /content/export_bundle.py")
if ret != 0:
    raise RuntimeError(f"Export failed with return code {ret}")

# Zip together
os.system("zip -j /content/namma_space_assets.zip /content/scene.splat /content/cameras.json")

# =============================================================================
# STAGE 4: Automated Download to Local Machine
# =============================================================================
print("\n=================================================================")
print("  SUCCESS: 3D Twin Generation Complete!")
print("  Downloading namma_space_assets.zip to your computer...")
print("=================================================================\n")
files.download("/content/namma_space_assets.zip")
